In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2
Configuration: {'general': {'run_name': 'experiment_with_4_classes', 'seed': 42, 'n_classes': 4}, 'dataset': {'split_type': 'test'}, 'paths': {'data_exploration_dir': 'output/experiment_with_4_classes/data_exploration', 'embeddings_dir': 'output/experiment_with_4_classes/embeddings', 'models_dir': 'output/experiment_with_4_classes/models', 'predictions_dir': 'output/experiment_with_4_classes/predictions', 'results_dir': 'output/experiment_with_4_classes/results'}, 'model': {'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False}, 'training': {'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3'}, 'evaluation': {'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}}

=== Configuration Variables ===

[DATASET]
  DATASET_SPLIT_TYPE: test

[EVALUATION]
  EVALUATIO

# Build Embeddings
This notebook generates embeddings for both SBERT (Sequence encoder) and OpenAI models and stores the indices inside the experiment folder.

In [2]:

import pathlib, json, numpy as np, faiss
from tqdm import tqdm
from src.datasets.dataset import get_dataset
from src.rag.vector_store import VectorStore
from src.rag import _EMBEDDINGS_DIR, _SBERT_DIR, _OPENAI_DIR
from src.embeddings.openai_embedder import OpenAIEmbedder
from sentence_transformers import SentenceTransformer

# ---- Parameters ----
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OPENAI_MODEL = "text-embedding-3-small"

print('Embeddings root:', _EMBEDDINGS_DIR)
_SBERT_DIR.mkdir(parents=True, exist_ok=True)
_OPENAI_DIR.mkdir(parents=True, exist_ok=True)


INFO | Loading faiss with AVX2 support.
INFO | Successfully loaded faiss with AVX2 support.
INFO | Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.
/home/marcmaceira/projects/reuters-rag-classifier_clean_v2/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embeddings root: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_4_classes/embeddings


In [3]:

# Load dataset
# X_train, y_train, _, _, _ = get_dataset(split_type="standard", n_classes=N_CLASSES)
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES
)

print(f"Loaded {len(X_train)} training documents with {N_CLASSES} classes")


TypeError: get_dataset() got an unexpected keyword argument 'cutoff_year'

In [ ]:

# ---- SBERT embeddings ----
sbert = SentenceTransformer(SBERT_MODEL)
vectors = sbert.encode(X_train, batch_size=64, show_progress_bar=True, convert_to_numpy=True).astype('float32')

meta = []
for i, (txt, label, vec) in enumerate(zip(X_train, y_train, vectors)):
    meta.append({"id": i, "label": label, "text": txt, "vector": vec.tolist()})

faiss_path = _SBERT_DIR / "index.faiss"
meta_path  = _SBERT_DIR / "meta.jsonl"
VectorStore.build(vectors, meta, vectors.shape[1], faiss_path, meta_path)
print("✅ SBERT index saved at", faiss_path)


In [ ]:

# ---- OpenAI embeddings ----
# Requires OPENAI_API_KEY env var
openai_embedder = OpenAIEmbedder(model=OPENAI_MODEL, batch_size=50)
openai_vecs = openai_embedder.encode(X_train)
openai_vecs = np.array(openai_vecs, dtype='float32')

meta_openai = []
for i, (txt, label, vec) in enumerate(zip(X_train, y_train, openai_vecs)):
    meta_openai.append({"id": i, "label": label, "text": txt, "vector": vec.tolist()})

openai_faiss = _OPENAI_DIR / "index.faiss"
openai_meta  = _OPENAI_DIR / "meta.jsonl"
VectorStore.build(openai_vecs, meta_openai, openai_vecs.shape[1], openai_faiss, openai_meta)
print("✅ OpenAI index saved at", openai_faiss)
